# 03 — Kalibrasi Peluang dan Pemilihan Threshold

**Kelompok 12 · Kelas B2 · Prediksi Customer Churn Telco**

`class_weight="balanced"` membuat peluang churn dari model inti cenderung terlalu tinggi. Notebook ini
mengkalibrasi peluang tersebut dengan metode sigmoid, memilih threshold keputusan hanya dari data latih,
lalu mengevaluasi hasil akhirnya pada data uji.

Output: `calibrator.pkl`, `threshold_selection.csv`, `calib_info.json`, `deployment_info.json`.
Jalankan setelah `02_Training_Model.ipynb`.

## 1. Import library

`CalibratedClassifierCV` untuk kalibrasi, `cross_val_predict` untuk peluang out-of-fold, dan
`high_risk_boundary` dari `prediction.py` agar batas risiko tinggi sama dengan aplikasi.

In [1]:
import hashlib
import json

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from prediction import high_risk_boundary
from training_data import BASE, classification_metrics, load_training_split

## 2. Memuat data dan model inti

Pembagian data sama dengan notebook 02. Model inti dibaca dari `model_churn.pkl`.

In [2]:
X_train, X_test, y_train, y_test = load_training_split()
base = joblib.load(BASE / "model_churn.pkl")
clf_name = type(base.named_steps["clf"]).__name__
print(f"Model inti: {clf_name} | data latih {len(X_train):,} | data uji {len(X_test):,}")

Model inti: LogisticRegression | data latih 5,625 | data uji 1,407


## 3. Menyiapkan kalibrator sigmoid

`CalibratedClassifierCV` melatih model inti pada 5 bagian data latih dan memetakan skornya ke peluang yang
lebih sesuai dengan kenyataan memakai fungsi sigmoid. `inner_cv` dipakai di dalam kalibrator, sedangkan
`outer_cv` dipakai untuk menghasilkan peluang out-of-fold pada langkah berikutnya.

In [3]:
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=43)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cal = CalibratedClassifierCV(estimator=base, method="sigmoid", cv=inner_cv)

## 4. Peluang out-of-fold (OOF) pada data latih

Setiap pelanggan latih diberi peluang churn oleh kalibrator yang tidak melihat pelanggan tersebut saat
dilatih. Peluang ini dipakai untuk memilih threshold tanpa menyentuh data uji.

In [4]:
oof = cross_val_predict(cal, X_train, y_train, cv=outer_cv,
                        method="predict_proba", n_jobs=-1)[:, 1]
print(f"Peluang OOF: {len(oof):,} pelanggan, rata-rata {oof.mean():.4f}")

Peluang OOF: 5,625 pelanggan, rata-rata 0.2657


## 5. Mencari threshold dengan F1 tertinggi

Threshold dicoba dari 0,10 sampai 0,60 dengan langkah 0,01. Threshold dengan F1 OOF tertinggi dipakai
aplikasi; seluruh hasil pencarian disimpan ke `threshold_selection.csv`.

In [5]:
thresholds = np.round(np.arange(0.10, 0.601, 0.01), 2)
scores = [f1_score(y_train, oof >= t, zero_division=0) for t in thresholds]
threshold = float(thresholds[int(np.argmax(scores))])
threshold_table = pd.DataFrame({"Threshold": thresholds, "OOF_F1": scores})
threshold_table.to_csv(BASE / "threshold_selection.csv", index=False)
print(f"Threshold OOF-train: {threshold:.2f} | OOF tuning F1: {max(scores):.4f}")
display(threshold_table.sort_values("OOF_F1", ascending=False).head(5).round(4))

Threshold OOF-train: 0.32 | OOF tuning F1: 0.6338


,Threshold,OOF_F1
22,0.32,0.6338
21,0.31,0.6321
20,0.30,0.6317
18,0.28,0.6317
23,0.33,0.6315


## 6. Melatih kalibrator akhir

Kalibrator dilatih pada seluruh data latih dan disimpan ke `calibrator.pkl`. File inilah yang dipakai
aplikasi untuk menghitung peluang churn.

In [6]:
cal.fit(X_train, y_train)
joblib.dump(cal, BASE / "calibrator.pkl", protocol=4)
print("Kalibrator disimpan ke calibrator.pkl")

Kalibrator disimpan ke calibrator.pkl


## 7. Evaluasi pada data uji

Peluang dari model inti (`p_base`) dan dari kalibrator (`p_cal`) dihitung pada 1.407 data uji. Tabel
membandingkan metrik kalibrator pada threshold terpilih dan pada threshold 0,50.

In [7]:
p_base = base.predict_proba(X_test)[:, 1]
p_cal = cal.predict_proba(X_test)[:, 1]
metrics = classification_metrics(y_test, p_cal, threshold)
metrics_at_0_5 = classification_metrics(y_test, p_cal, 0.5)
display(pd.DataFrame([metrics, metrics_at_0_5], index=[f"Threshold {threshold:.2f}", "Threshold 0.50"]))

,Accuracy,Precision,Recall,F1,ROC_AUC,Brier,TN,FP,FN,TP
Threshold 0.32,0.7569,0.5309,0.7353,0.6166,0.8324,0.1414,790,243,99,275
Threshold 0.50,0.8024,0.6472,0.5642,0.6029,0.8324,0.1414,918,115,163,211


## 8. Keandalan peluang per tingkat risiko

Data uji dikelompokkan menjadi risiko rendah, sedang, dan tinggi. Pada setiap kelompok, rata-rata peluang
prediksi dibandingkan dengan persentase churn yang sebenarnya; semakin dekat, semakin dapat dipercaya.

In [8]:
high = high_risk_boundary(threshold)
bands = [(0.0, threshold, "LOW"), (threshold, high, "MID"), (high, 1.01, "HIGH")]
reliability = []
for lo, hi, label in bands:
    mask = (p_cal >= lo) & (p_cal < hi)
    reliability.append({
        "band": label, "n": int(mask.sum()),
        "mean_pred": round(float(p_cal[mask].mean()), 4) if mask.any() else None,
        "empirical": round(float(y_test[mask].mean()), 4) if mask.any() else None,
    })
display(pd.DataFrame(reliability))

,band,n,mean_pred,empirical
0,LOW,889,0.1068,0.1114
1,MID,303,0.4613,0.4257
2,HIGH,215,0.6872,0.6791


## 9. Menyimpan `calib_info.json`

Brier score mengukur rata-rata kesalahan kuadrat peluang (semakin kecil semakin baik). Nilai sebelum dan
sesudah kalibrasi disimpan bersama tabel keandalan.

In [9]:
calib_info = {
    "method": "sigmoid", "cv": 5, "fit_on": "train-only", "base_model": clf_name,
    "brier_base": round(float(brier_score_loss(y_test, p_base)), 4),
    "brier_cal": round(float(brier_score_loss(y_test, p_cal)), 4),
    "reliability_test": reliability, "n_train": len(X_train), "n_test": len(X_test),
}
(BASE / "calib_info.json").write_text(json.dumps(calib_info, indent=2), encoding="utf-8")
print(f"Brier score: {calib_info['brier_base']} -> {calib_info['brier_cal']}")

Brier score: 0.1741 -> 0.1414


## 10. Menyimpan `deployment_info.json`

Berisi threshold, batas risiko tinggi, metrik akhir, dan hash SHA-256 kedua file model. Saat dibuka, aplikasi
mencocokkan hash ini dengan file model agar model yang dipakai sama dengan model yang dievaluasi.

In [10]:
def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


deployment = {
    "base_model": clf_name, "probability_model": "sigmoid calibrated ensemble (5-fold)",
    "threshold": threshold, "threshold_selection": "max F1 on 5-fold OOF train probabilities",
    "threshold_selection_data": "train-only", "oof_tuning_f1": round(float(max(scores)), 4),
    "risk_high_boundary": high, "n_train": len(X_train), "n_test": len(X_test),
    "n_features": X_train.shape[1], "sklearn_version": sklearn.__version__,
    "metrics": metrics, "metrics_at_0_5": metrics_at_0_5,
    "model_sha256": file_hash(BASE / "model_churn.pkl"),
    "calibrator_sha256": file_hash(BASE / "calibrator.pkl"),
}
(BASE / "deployment_info.json").write_text(json.dumps(deployment, indent=2), encoding="utf-8")
print("Evaluasi model aplikasi:", deployment["metrics"])

Evaluasi model aplikasi: {'Accuracy': 0.7569, 'Precision': 0.5309, 'Recall': 0.7353, 'F1': 0.6166, 'ROC_AUC': 0.8324, 'Brier': 0.1414, 'TN': 790, 'FP': 243, 'FN': 99, 'TP': 275}
